## Saving the Final Model

The final trained XGBoost model is saved using `joblib` to ensure reproducibility and enable future forecasting without retraining the model.

Saving the model provides several benefits:

- Avoids retraining the model every time predictions are needed.
- Preserves the exact model configuration and learned patterns.
- Facilitates deployment and future inference.
- Ensures reproducibility of forecasting results.

Before saving, the trained model object must exist in the current notebook session. If the model was trained in a previous notebook or the kernel was restarted, the model should first be loaded into memory.

---

## Why Single-Zone Forecasting Was Selected

The objective of this project was to develop an accurate and deployable taxi demand forecasting system using historical NYC taxi trip data. Although the dataset contains multiple taxi zones, a **single-zone forecasting approach** was selected for the following reasons:

### 1. Limited Historical Data Availability

The dataset consists of only **one month of data (January 2026)**. For each individual zone, this corresponds to approximately **744 hourly observations (31 days × 24 hours)**.

Building a reliable multi-zone forecasting model typically requires several months or years of historical data to capture:

- Seasonal patterns
- Weekly trends
- Zone-specific demand variations
- Holiday and special event effects

Given the limited time span, a single-zone approach provides a more robust forecasting framework.

---

### 2. Focus on Forecasting Accuracy

The primary objective of this project was to develop a model capable of producing **accurate hourly demand forecasts**.

By focusing on a single zone:

- The model learns demand patterns specific to that zone.
- Feature engineering becomes more meaningful.
- Recursive forecasting becomes more stable.
- Forecast evaluation is easier to interpret.

This improves overall model reliability compared to a generalized model trained across all zones.

---

### 3. Reduced Model Complexity

Multi-zone forecasting introduces additional complexity, including:

- Zone-wise lag feature generation
- Zone-specific rolling statistics
- Handling varying demand distributions across zones
- More complex deployment pipelines

Since the purpose of this project was to demonstrate an **end-to-end forecasting workflow**, a single-zone model provided a simpler and more maintainable solution.

---

### 4. Selection of the Most Active Zone

The forecasting zone was selected using the following approach:

```python
zone_counts = (
    df.groupby("PULocationID")
    .size()
    .sort_values(ascending=False)
)

zone_id = zone_counts.index[0]
```

In [4]:
import joblib

model = joblib.load(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/Yellow_Dataset/models/xgboost_model.pkl"
)

print(
    "XGBoost model loaded successfully."
)

XGBoost model loaded successfully.


In [5]:
joblib.dump(
    model,
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/Yellow_Dataset/models/final_xgboost_model.pkl"
)

print(
    "Final XGBoost model saved successfully."
)

Final XGBoost model saved successfully.


Step 3: Verify Model Loaded Correctly (Recommended)

In [6]:
print(type(model))

<class 'xgboost.sklearn.XGBRegressor'>


Step 5: Verify Final Model Exists (Optional)

In [7]:
import os

model_path = (
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/Yellow_Dataset/models/final_xgboost_model.pkl"
)

print(
    "Model Exists:",
    os.path.exists(model_path)
)

Model Exists: True


## Load Existing Validation Forecast Results

The validation forecast results were generated and saved during the recursive forecasting notebook. To avoid duplication and maintain a modular workflow, the existing results are loaded directly from disk instead of regenerating them.

This approach ensures:

- Consistency across notebooks.
- Reduced computational overhead.
- Better separation of forecasting and deployment tasks.
- Improved reproducibility of the project pipeline.

In [8]:
import pandas as pd

forecast_results = pd.read_csv(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/Yellow_Dataset/results/validation_forecast_results.csv"
)

print(
    "Validation forecast results loaded successfully."
)

forecast_results.head()

Validation forecast results loaded successfully.


,hour,actual_demand,forecasted_demand,residual
0,2026-01-25 01:00:00,614,609.482500,4.517517
1,2026-01-25 02:00:00,448,481.816300,-33.816315
2,2026-01-25 03:00:00,269,359.517180,-90.517181
3,2026-01-25 04:00:00,72,113.945335,-41.945335
4,2026-01-25 05:00:00,18,34.369250,-16.369251


In [ ]:
performance_summary = pd.DataFrame({

    "Model": ["XGBoost"],

    "Validation_MAE": [91.87],

    "Validation_RMSE": [121.27]

})

performance_summary.to_csv(

    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/Yellow_Dataset/results/final_model_performance.csv",

    index=False

)

print(
    "Model performance summary saved successfully."
)

Model performance summary saved successfully.


In [4]:
import pandas as pd
hourly_df = pd.read_parquet(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/Yellow_Dataset/data/hourly_demand.parquet"
)

zone_79 = (
    hourly_df[
        hourly_df["PULocationID"] == 79
    ]
    .sort_values("hour")
)

print(
    "Zone 79 Shape:",
    zone_79.shape
)

print(
    zone_79.head()
)

print(
    zone_79.tail()
)

Zone 79 Shape: (745, 3)
                   hour  PULocationID  demand
59  2026-01-01 00:00:00            79     432
268 2026-01-01 01:00:00            79     459
485 2026-01-01 02:00:00            79     560
691 2026-01-01 03:00:00            79     576
892 2026-01-01 04:00:00            79     326
                      hour  PULocationID  demand
122936 2026-01-31 20:00:00            79     354
123111 2026-01-31 21:00:00            79     445
123292 2026-01-31 22:00:00            79     583
123473 2026-01-31 23:00:00            79     732
123601 2026-02-01 00:00:00            79       1
